# Cells for a model

A model works on cells (1 km squares, say), not on 100 m pixels. `egms_client.cells`
gives each cell what its pixels say, with three rules a model easily breaks:

1. **Not measured is not zero.** A cell with no pixel gets `None` (NaN in a table), never 0.
2. **Signs are kept.** Up and down stay apart, and so do a slope speeding up and one
   slowing down.
3. **One mask.** Every statistic uses the same pixels: those with a velocity.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# The crops of the benchmark: small pieces of the real archive, in the repository.
# No key and no download: this notebook runs anywhere, and in CI.
DATA = Path("../benchmarks/data")
IMAGES = Path("../docs/images")  # the figure is saved there when the folder exists

In [ ]:
import rasterio

from egms_client.cells import on_grid, per_cell, square_cells, to_frame
from egms_client.mosaic import mosaic


def cells_of(crop: str, size_m: float = 1_000) -> pd.DataFrame:
    """Per-cell statistics of a crop, on the cells fully inside it."""
    folder = DATA / crop
    with rasterio.open(folder / "U.tif") as src:
        left, bottom, right, top = src.bounds
    box = {"minx": left, "miny": bottom, "maxx": right, "maxy": top}
    up = mosaic([folder / "U.tif"], **box)
    east = mosaic([folder / "E.tif"], **box) if (folder / "E.tif").exists() else None
    attributes = pd.read_parquet(folder / "l3.parquet")
    cells = [
        c
        for c in square_cells(**box, size_m=size_m)
        if c.minx >= left and c.maxx <= right and c.miny >= bottom and c.maxy <= top
    ]
    stats = per_cell(
        cells,
        up,
        east,
        acceleration=on_grid(attributes, "acceleration", up),
        seasonality=on_grid(attributes, "seasonality", up),
        rmse=on_grid(attributes, "rmse_ts", up),
    )
    return to_frame(stats)


forest = cells_of("foresta-umbra")
forest[["id", "pixels", "valid_pixels", "vertical_median", "score", "reasons"]].head(8)

In [ ]:
share = (forest.valid_pixels == 0).mean()
print(f"Foresta Umbra: {share:.0%} of the {len(forest)} kilometres have no pixel.")
print("Written as 0.0, they would be 'still ground' for a model. Here they are NaN.")
print(
    f"vertical_median is NaN in {forest.vertical_median.isna().sum()} rows, 0.0 in {(forest.vertical_median == 0).sum()}."
)

## The sign of the acceleration

At the Murge, a place that does not move, the signed vertical acceleration of the cells
sits around zero: about as many negative as positive. The magnitude `sqrt(E² + U²)`, as a
model that only looks at "how much" would compute it, is always positive: it turns noise
into a signal.

In [ ]:
from egms_client.mosaic import Mosaic

murge = cells_of("murge")
attributes_e = pd.read_parquet(DATA / "murge" / "l3-E.parquet")
attributes_u = pd.read_parquet(DATA / "murge" / "l3.parquet")
with rasterio.open(DATA / "murge" / "U.tif") as src:
    left, bottom, right, top = src.bounds
box = {"minx": left, "miny": bottom, "maxx": right, "maxy": top}
up = mosaic([DATA / "murge" / "U.tif"], **box)
a_u = on_grid(attributes_u, "acceleration", up)
a_e = on_grid(attributes_e, "acceleration", mosaic([DATA / "murge" / "E.tif"], **box))
# The p90 of the magnitude per km: what a model that only asks "how much" would use.
magnitude = Mosaic(np.sqrt(a_u**2 + a_e**2), up.left, up.top)
inside = [
    c
    for c in square_cells(**box)
    if c.minx >= left and c.maxx <= right and c.miny >= bottom and c.maxy <= top
]
p90 = pd.Series([s.vertical_p90 for s in per_cell(inside, magnitude)], dtype="float64")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].hist(murge.acceleration_median.dropna(), bins=30, color="#4292c6")
axes[0].axvline(0, color="k", lw=1)
axes[0].set_title(f"signed median per km: {murge.acceleration_median.median():+.2f}")
axes[0].set_xlabel("vertical acceleration (mm/year²)")
axes[0].set_ylabel("km cells")
axes[1].hist(p90.dropna(), bins=30, color="#cb181d")
axes[1].axvline(0, color="k", lw=1)
axes[1].set_title(f"p90 of the magnitude per km: median {p90.median():.2f}")
axes[1].set_xlabel("sqrt(E² + U²) of the acceleration (mm/year²)")
print(
    f"{len(murge)} km at the Murge: {(murge.acceleration_median < 0).sum()} with a negative signed median, "
    f"{(murge.acceleration_median > 0).sum()} positive; the magnitude is positive in all {p90.notna().sum()}."
)
if IMAGES.exists():
    fig.savefig(IMAGES / "notebook-acceleration.png", dpi=110, bbox_inches="tight")

## Sinking, but slower

In the Bologna box the median cell sinks, and its acceleration is positive: the sinking
slows down. A magnitude would call it "speeding up".

In [ ]:
bologna = cells_of("bologna")
print(f"median vertical velocity: {bologna.vertical_median.median():+.1f} mm/year")
print(f"median vertical acceleration: {bologna.acceleration_median.median():+.2f} mm/year²")
sinking = bologna[bologna.vertical_median < -2]
slower = (sinking.acceleration_median > 0).mean()
print(
    f"{len(sinking)} km sink faster than 2 mm/year; in {slower:.0%} of them the sinking slows down."
)

**What is not there.** The east-west acceleration is not kept per cell (only the vertical
one, signed). `valid_pixels` counts 100 m pixels, not radar targets. Cells cut by the edge
of a crop count only the pixels inside. The slowdown at Bologna is what the data say: it
is not yet checked against a source.

*Data: EGMS, © European Union, Copernicus Land Monitoring Service.*